# ScanImage structural volume alignment QC: BiDi, repeat averaging, crop-based z registration

This notebook runs the ScanImage branch of `slap2_volume_align` in three stages:

1. **BiDi correction** on raw ScanImage pages, before any registration.
2. **Within-plane repeat alignment/averaging**: repeated frames from the same z plane are registered to a same-plane template and averaged.
3. **Crop-based z-plane registration**: the averaged z volume is aligned across planes using one or more local crops/landmarks, replacing the older global full-frame z-alignment path.

The z step is deliberately crop/local because sparse neuronal morphology is usually not well modeled by a single full-frame phase-correlation trajectory across z. Manual crops around stable somata, dendrite bundles, ablation features, or other bright landmarks are preferred for final outputs. Auto-crops are useful for smoke tests.

## 1. Imports and repo setup

Run from the repo root or any notebook subdirectory. The cell below adds `src/` to `sys.path` so the local editable repo is imported rather than an older installed copy.

In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path
from typing import Optional

from IPython.display import display, HTML

import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy import ndimage


def find_repo_root(start: Optional[Path] = None) -> Path:
    if start is None:
        start = Path.cwd()
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "slap2_volume_align").exists():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Repo root:", REPO_ROOT)
print("Package source:", SRC)

In [ ]:
%load_ext autoreload
%autoreload 2

try:
    %matplotlib widget
except Exception:
    %matplotlib notebook

In [ ]:
from slap2_volume_align.readers.tiff import read_tiff_stack_spec, read_plane_channel_frames
from slap2_volume_align.sources.scanimage.metadata import parse_scanimage_description
from slap2_volume_align.sources.scanimage.pipeline import ScanImageAverageConfig, average_scanimage_volume
from slap2_volume_align.core.bidirectional import estimate_bidirectional_phase_stack, apply_bidirectional_phase
from slap2_volume_align.core.z_registration import (
    infer_registration_crops,
    estimate_crop_z_registration,
    apply_z_registration,
    register_volume_z_tiff,
    make_z_registration_qc_png,
    write_z_registration_csv,
)

## 2. User parameters

Start with a small z range if you are testing a new TIFF. For final processing, set `PLANE_STOP = None`.

For z registration, either leave `Z_REGISTRATION_CROPS_YX = None` to auto-detect bright crops, or provide one or more manual crops as `(y0, y1, x0, x1)` in full-resolution pixel coordinates. Manual crops are strongly recommended for final figures/data products.

In [ ]:
# --- Input/output paths ---
INPUT_TIF = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\ASAP8\853264\ablation\2026-07-07_853264_ablation\ablation_evidence_00002.tif")
OUT_DIR = INPUT_TIF.parent / "aligned_volume_crop_zreg"

# --- Stack layout ---
N_PLANES = 50
REPEATS_PER_PLANE = 20
N_CHANNELS = 1
ALIGNMENT_CHANNEL = 0  # zero-based; 0 == channel 1
ORDER = "slice_blocks"  # "slice_blocks" or "volume_interleaved"

# --- Processing range ---
PLANE_START = 0
PLANE_STOP = None  # e.g. 5 for smoke test, None for all planes

# --- BiDi correction ---
AUTO_ESTIMATE_BIDIPHASE = True
BIDI_PHASE_CANDIDATES = np.arange(-10.0, 10.01, 0.25)
BIDI_LINE_PARITY = "auto"  # "auto", "odd", or "even"
BIDI_SHIFT_MODE = "symmetric"  # "symmetric" usually reduces sawtooth texture
BIDI_FILL_MODE = "nearest"
BIDI_SAMPLE_Z_INDICES = [0, 8, 16, 24, 32, 40, 49]
BIDI_SAMPLE_REPEAT_INDICES = [0, 5, 10, 15]

# --- Within-plane repeat alignment/averaging ---
ALIGN_REPEATS = True
REGISTRATION_BINNING = 2
UPSAMPLE_FACTOR = 20
MAX_SHIFT_PX = 12
HIGHPASS_SIGMA_PX = 8.0
REGISTRATION_N_PASSES = 2
INTERPOLATION_ORDER = 1
OUTPUT_DTYPE = "float32"
OUTPUT_COMPRESSION = None  # None is fastest; "zlib" saves space

# --- Crop-based z-plane registration ---
REGISTER_Z = True
# Manual examples: [(420, 850, 520, 950)] or [(420, 850, 520, 950), (900, 1250, 1200, 1550)]
Z_REGISTRATION_CROPS_YX = None
Z_ANCHOR = None
Z_AUTO_N_CROPS = 1
Z_AUTO_CROP_SIZE_PX = 512
Z_TEMPLATE_RADIUS = 2
Z_REGISTRATION_BINNING = 4
Z_MAX_SHIFT_PX = 24
Z_HIGHPASS_SIGMA_PX = 8.0
Z_INTENSITY_TRANSFORM = "sqrt"  # "sqrt", "log1p", or "none"
Z_MIN_CORR = 0.06
Z_MIN_OVERLAP_FRACTION = 0.25
Z_MIN_ACCEPTED_CROPS = 1
Z_SMOOTH_WINDOW = 0  # 0 disables smoothing; try 3/5 only after raw shifts look sensible
Z_MAX_INTERPOLATION_GAP = 1

print(INPUT_TIF)
print(OUT_DIR)

## 3. Inspect the source TIFF without loading it into memory

This checks page count, shape, dtype, and a few ScanImage metadata fields. It also helps verify whether the TIFF is `slice_blocks` or `volume_interleaved`.

In [ ]:
with tifffile.TiffFile(INPUT_TIF) as tif:
    print(f"n_pages: {len(tif.pages)}")
    print(f"page[0].shape: {tif.pages[0].shape}")
    print(f"page[0].dtype: {tif.pages[0].dtype}")
    print("\nFirst page description excerpt:")
    print((tif.pages[0].description or "")[:1500])

    probe_pages = sorted(set([
        0, 1, REPEATS_PER_PLANE - 1, REPEATS_PER_PLANE,
        2 * REPEATS_PER_PLANE - 1, 2 * REPEATS_PER_PLANE,
        len(tif.pages) - 1,
    ]))
    rows = []
    for p in probe_pages:
        if 0 <= p < len(tif.pages):
            parsed = parse_scanimage_description(tif.pages[p].description)
            rows.append({
                "page": p,
                "frame_number": parsed.frame_number,
                "acquisition_number": parsed.acquisition_number,
                "frame_number_acquisition": parsed.frame_number_acquisition,
                "frame_timestamp_sec": parsed.frame_timestamp_sec,
            })

pd.DataFrame(rows)

## 4. Verify inferred layout

`read_tiff_stack_spec` validates that the requested layout accounts for every TIFF page. If this is wrong, do not proceed; the averaging step would combine the wrong pages.

In [ ]:
spec = read_tiff_stack_spec(
    INPUT_TIF,
    n_planes=N_PLANES,
    repeats_per_plane=REPEATS_PER_PLANE,
    n_channels=N_CHANNELS,
    order=ORDER,
    infer_from_descriptions=True,
)

spec.to_dict()

## 5. Raw page-order/contact-sheet QC

Tiles within a row should look like repeated acquisitions of the same z plane if `ORDER` is correct.

In [ ]:
def robust_limits(img, pct=(1, 99.8)):
    finite = np.isfinite(img)
    if not finite.any():
        return 0, 1
    lo, hi = np.percentile(img[finite], pct)
    if hi <= lo:
        hi = lo + 1
    return lo, hi


def show_raw_repeats(input_tif, z_indices=(0, 1, 2), channel_index=0, max_repeats=8, downsample=8):
    nrows = len(z_indices)
    ncols = min(max_repeats, spec.repeats_per_plane)
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.2 * ncols, 2.2 * nrows), squeeze=False)
    with tifffile.TiffFile(input_tif) as tif:
        for r, z in enumerate(z_indices):
            frames = read_plane_channel_frames(tif, z_index=int(z), channel_index=channel_index, spec=spec)
            for c in range(ncols):
                ax = axes[r, c]
                img = frames[c][::downsample, ::downsample]
                lo, hi = robust_limits(img)
                ax.imshow(img, cmap="gray", vmin=lo, vmax=hi)
                ax.set_title(f"z {z}, rep {c}", fontsize=8)
                ax.axis("off")
    fig.tight_layout()
    return fig

show_raw_repeats(INPUT_TIF, z_indices=[0, min(10, spec.n_planes-1), min(25, spec.n_planes-1)], max_repeats=8, downsample=8);

## 6. Estimate BiDi phase before alignment

BiDi correction should happen first, per plane/repeat, before within-plane registration or z registration.

In [ ]:
if AUTO_ESTIMATE_BIDIPHASE:
    sample_frames = []
    sample_meta = []
    candidate_z = [z for z in BIDI_SAMPLE_Z_INDICES if 0 <= int(z) < spec.n_planes]
    candidate_repeats = [r for r in BIDI_SAMPLE_REPEAT_INDICES if 0 <= int(r) < spec.repeats_per_plane]

    with tifffile.TiffFile(INPUT_TIF) as tif:
        for z in candidate_z:
            frames = read_plane_channel_frames(tif, z_index=int(z), channel_index=ALIGNMENT_CHANNEL, spec=spec)
            for r in candidate_repeats:
                sample_frames.append(frames[int(r)])
                sample_meta.append({"z_index": int(z), "repeat_index": int(r)})

    parity_candidates = ("odd", "even") if BIDI_LINE_PARITY == "auto" else (BIDI_LINE_PARITY,)
    bidi_estimate = estimate_bidirectional_phase_stack(
        sample_frames,
        phase_candidates=BIDI_PHASE_CANDIDATES,
        line_parity_candidates=parity_candidates,
        crop_yx=(128, 128),
        x_stride=1,
        highpass_sigma_px=8.0,
        shift_mode=BIDI_SHIFT_MODE,
    )

    BIDIPHASE = float(bidi_estimate["best_phase"])
    BIDI_LINE_PARITY = str(bidi_estimate["best_line_parity"])
    display({k: v for k, v in bidi_estimate.items() if k != "per_frame_scores"})
else:
    BIDIPHASE = 0.0

print("BIDIPHASE:", BIDIPHASE)
print("BIDI_LINE_PARITY:", BIDI_LINE_PARITY)
print("BIDI_SHIFT_MODE:", BIDI_SHIFT_MODE)

## 7. Optional visual BiDi sanity check

This compares raw vs BiDi-corrected crops for one representative frame.

In [ ]:
with tifffile.TiffFile(INPUT_TIF) as tif:
    frame0 = read_plane_channel_frames(
        tif,
        z_index=int(BIDI_SAMPLE_Z_INDICES[min(1, len(BIDI_SAMPLE_Z_INDICES)-1)]),
        channel_index=ALIGNMENT_CHANNEL,
        spec=spec,
    )[0]

corrected0 = apply_bidirectional_phase(
    frame0,
    BIDIPHASE,
    line_parity=BIDI_LINE_PARITY,
    fill_mode=BIDI_FILL_MODE,
    shift_mode=BIDI_SHIFT_MODE,
)

crop = np.s_[frame0.shape[0]//2-256:frame0.shape[0]//2+256, frame0.shape[1]//2-256:frame0.shape[1]//2+256]
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, img, title in zip(axes, [frame0[crop], corrected0[crop]], ["raw", "BiDi corrected"]):
    lo, hi = robust_limits(img)
    ax.imshow(img, cmap="gray", vmin=lo, vmax=hi)
    ax.set_title(title)
    ax.axis("off")
fig.tight_layout()

## 8. Run ScanImage averaging and optional crop-based z registration

This writes:

- `*_avg_chN.tif`: BiDi-corrected, within-plane repeat-aligned/averaged volume.
- `*_avg_chN_zregistered.tif`: optional crop-based z-registered volume.
- CSVs for repeat shifts and z-registration shifts.
- QC PNGs.

In [ ]:
config = ScanImageAverageConfig(
    input_tif=INPUT_TIF,
    out_dir=OUT_DIR,
    n_planes=N_PLANES,
    repeats_per_plane=REPEATS_PER_PLANE,
    n_channels=N_CHANNELS,
    alignment_channel=ALIGNMENT_CHANNEL,
    order=ORDER,
    plane_start=PLANE_START,
    plane_stop=PLANE_STOP,
    align=ALIGN_REPEATS,
    registration_binning=REGISTRATION_BINNING,
    upsample_factor=UPSAMPLE_FACTOR,
    max_shift_px=MAX_SHIFT_PX,
    highpass_sigma_px=HIGHPASS_SIGMA_PX,
    interpolation_order=INTERPOLATION_ORDER,
    output_dtype=OUTPUT_DTYPE,
    output_compression=OUTPUT_COMPRESSION,
    bidiphase=BIDIPHASE,
    bidi_line_parity=BIDI_LINE_PARITY,
    bidi_fill_mode=BIDI_FILL_MODE,
    bidi_shift_mode=BIDI_SHIFT_MODE,
    registration_n_passes=REGISTRATION_N_PASSES,
    register_z=REGISTER_Z,
    z_registration_crops_yx=Z_REGISTRATION_CROPS_YX,
    z_anchor=Z_ANCHOR,
    z_auto_n_crops=Z_AUTO_N_CROPS,
    z_auto_crop_size_px=Z_AUTO_CROP_SIZE_PX,
    z_template_radius=Z_TEMPLATE_RADIUS,
    z_registration_binning=Z_REGISTRATION_BINNING,
    z_max_shift_px=Z_MAX_SHIFT_PX,
    z_highpass_sigma_px=Z_HIGHPASS_SIGMA_PX,
    z_intensity_transform=Z_INTENSITY_TRANSFORM,
    z_min_corr=Z_MIN_CORR,
    z_min_overlap_fraction=Z_MIN_OVERLAP_FRACTION,
    z_min_accepted_crops=Z_MIN_ACCEPTED_CROPS,
    z_smooth_window=Z_SMOOTH_WINDOW,
    z_max_interpolation_gap=Z_MAX_INTERPOLATION_GAP,
)

summary = average_scanimage_volume(config)
summary

## 9. Load summary and inspect repeat-alignment shifts

A healthy within-plane repeat registration should usually have nonzero but plausible shifts. All zeros means either there was truly no detectable motion or the registration is not seeing enough signal.

In [ ]:
summary_path = Path(summary["summary_json"])
summary_loaded = json.loads(summary_path.read_text())
print(summary_path)

shift_df = pd.read_csv(summary_loaded["shifts_csv"])
display(shift_df.head())
print("n rows:", len(shift_df))
print("accepted fraction:", shift_df["accepted"].mean() if "accepted" in shift_df else np.nan)

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, key, ylabel in zip(axes[:2], ["shift_y_px", "shift_x_px"], ["repeat y shift (px)", "repeat x shift (px)"]):
    ax.plot(shift_df["z_index"], shift_df[key], ".", alpha=0.35)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
mag = np.hypot(shift_df["shift_y_px"], shift_df["shift_x_px"])
axes[2].plot(shift_df["z_index"], mag, ".", alpha=0.35)
axes[2].set_ylabel("repeat shift magnitude")
axes[2].set_xlabel("z index")
axes[2].grid(alpha=0.25)
fig.tight_layout()

## 10. Inspect averaged and z-registered outputs

The first montage shows the averaged volume. If z registration was enabled, the second montage shows the crop-based registered volume.

In [ ]:
def load_first_output(summary_loaded, key):
    paths = summary_loaded.get(key, {}) or {}
    if not paths:
        return None, None
    first_key = sorted(paths)[0]
    p = Path(paths[first_key])
    return p, tifffile.imread(p).astype(np.float32, copy=False)


def show_volume_montage(volume, title="volume", planes=None, max_planes=12, downsample=8):
    if planes is None:
        planes = np.linspace(0, volume.shape[0] - 1, min(max_planes, volume.shape[0]), dtype=int)
    n = len(planes)
    fig, axes = plt.subplots(1, n, figsize=(2.1 * n, 2.4), squeeze=False)
    for ax, z in zip(axes.ravel(), planes):
        img = volume[int(z), ::downsample, ::downsample]
        lo, hi = robust_limits(img)
        ax.imshow(img, cmap="gray", vmin=lo, vmax=hi)
        ax.set_title(f"z {int(z)}", fontsize=8)
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    return fig

avg_path, avg_vol = load_first_output(summary_loaded, "outputs")
zreg_path, zreg_vol = load_first_output(summary_loaded, "z_registered_outputs")
print("Averaged:", avg_path)
print("Z registered:", zreg_path)

show_volume_montage(avg_vol, title="averaged, not z-registered", downsample=8);
if zreg_vol is not None:
    show_volume_montage(zreg_vol, title="crop-based z registered", downsample=8);

## 11. Inspect z-registration crops and shift evidence

If many planes are unsupported, adjust the crop(s), anchor, or thresholds rather than forcing smoothing/extrapolation.

In [ ]:
z_csv = summary_loaded.get("z_registration_csv")
z_crop_csv = summary_loaded.get("z_registration_crop_csv")
print("z_registration_csv:", z_csv)
print("z_registration_crop_csv:", z_crop_csv)
print("crops_yx:", summary_loaded.get("z_registration_crops_yx"))

if z_csv:
    z_df = pd.read_csv(z_csv)
    display(z_df.head())
    print("accepted fraction:", z_df["accepted"].mean())
    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    ok = z_df["accepted"].astype(bool)
    axes[0].plot(z_df.loc[ok, "z_index"], z_df.loc[ok, "shift_y_px"], ".", label="accepted")
    axes[0].plot(z_df.loc[~ok, "z_index"], z_df.loc[~ok, "shift_y_px"], "x", label="unsupported")
    axes[0].set_ylabel("z-reg y shift")
    axes[0].legend()
    axes[1].plot(z_df.loc[ok, "z_index"], z_df.loc[ok, "shift_x_px"], ".", label="accepted")
    axes[1].plot(z_df.loc[~ok, "z_index"], z_df.loc[~ok, "shift_x_px"], "x", label="unsupported")
    axes[1].set_ylabel("z-reg x shift")
    axes[2].plot(z_df["z_index"], z_df["n_accepted_crops"], ".-")
    axes[2].set_ylabel("accepted crops")
    axes[2].set_xlabel("z index")
    for ax in axes:
        ax.grid(alpha=0.25)
    fig.tight_layout()
else:
    print("No z registration CSV. Set REGISTER_Z=True or run the posthoc cell below.")

## 12. Crop planning view

Use this on the averaged volume to pick manual crops. Coordinates are full-resolution pixels. After choosing crops, set `Z_REGISTRATION_CROPS_YX` in the parameter cell or use the posthoc registration cell below.

In [ ]:
def show_crop_planning_projection(volume, crops=None, downsample=4, pct=(1, 99.8)):
    proj = np.nanmax(volume, axis=0)
    display_img = proj[::downsample, ::downsample]
    lo, hi = robust_limits(display_img, pct)
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(display_img, cmap="gray", vmin=lo, vmax=hi)
    ax.set_title(f"Max projection for crop planning (display downsample={downsample})")
    ax.set_xlabel(f"x / {downsample}")
    ax.set_ylabel(f"y / {downsample}")
    if crops:
        for i, (y0, y1, x0, x1) in enumerate(crops):
            ax.add_patch(Rectangle((x0/downsample, y0/downsample), (x1-x0)/downsample, (y1-y0)/downsample, fill=False, lw=2))
            ax.text(x0/downsample, y0/downsample, f"crop {i}", color="white")
    return fig

current_crops = summary_loaded.get("z_registration_crops_yx") or Z_REGISTRATION_CROPS_YX
show_crop_planning_projection(avg_vol, crops=current_crops, downsample=4);
print("Current crops:", current_crops)
print("Manual crop example: Z_REGISTRATION_CROPS_YX = [(y0, y1, x0, x1)]")

## 13. Posthoc crop-based z registration tuning

Use this to iterate on crop(s), thresholds, and anchor without rerunning the slow repeat-level averaging step. It reads `*_avg_ch1.tif` and writes a new `*_zregistered.tif` plus CSV/QC files.

In [ ]:
RUN_POSTHOC_Z_REGISTRATION = False

POSTHOC_CROPS_YX = Z_REGISTRATION_CROPS_YX  # e.g. [(420, 850, 520, 950)]
POSTHOC_ANCHOR_Z = Z_ANCHOR
POSTHOC_SUFFIX = "manualcrop"

if RUN_POSTHOC_Z_REGISTRATION:
    posthoc_out = avg_path.with_name(f"{avg_path.stem}_zregistered_{POSTHOC_SUFFIX}.tif")
    posthoc_summary = register_volume_z_tiff(
        avg_path,
        output_tif=posthoc_out,
        crops_yx=POSTHOC_CROPS_YX,
        anchor_z=POSTHOC_ANCHOR_Z,
        auto_n_crops=Z_AUTO_N_CROPS,
        auto_crop_size_px=Z_AUTO_CROP_SIZE_PX,
        template_radius=Z_TEMPLATE_RADIUS,
        max_shift_px=Z_MAX_SHIFT_PX,
        binning=Z_REGISTRATION_BINNING,
        highpass_sigma_px=Z_HIGHPASS_SIGMA_PX,
        intensity_transform=Z_INTENSITY_TRANSFORM,
        min_corr=Z_MIN_CORR,
        min_overlap_fraction=Z_MIN_OVERLAP_FRACTION,
        min_accepted_crops=Z_MIN_ACCEPTED_CROPS,
        smooth_window=Z_SMOOTH_WINDOW,
        max_interpolation_gap=Z_MAX_INTERPOLATION_GAP,
        interpolation_order=INTERPOLATION_ORDER,
        output_dtype=OUTPUT_DTYPE,
        output_compression=OUTPUT_COMPRESSION,
    )
    display(posthoc_summary)
    tuned_vol = tifffile.imread(posthoc_summary["output_tif"]).astype(np.float32, copy=False)
    show_volume_montage(tuned_vol, title=f"posthoc z registered: {POSTHOC_SUFFIX}", downsample=8);
else:
    print("Set RUN_POSTHOC_Z_REGISTRATION=True to tune z registration without rerunning averaging.")

## 14. Synthetic sanity check for crop-based z registration

This cell simulates a small shifted z stack and verifies that the crop-based z registration recovers the imposed shifts. It is useful after repo edits or environment changes.

In [ ]:
def synthetic_plane(shape=(128, 128)):
    y, x = np.indices(shape)
    img = np.zeros(shape, dtype=np.float32)
    for cy, cx, sig, amp in [(42, 55, 3.0, 1.0), (72, 70, 5.0, 0.7), (55, 90, 2.5, 0.5)]:
        img += amp * np.exp(-((y-cy)**2 + (x-cx)**2) / (2 * sig**2))
    return img

base = synthetic_plane()
true_shifts = np.array([[0, 0], [2, -3], [4, -5], [1, -2], [-2, 3]], dtype=float)
sim_vol = np.stack([
    ndimage.shift(base, shift=(-dy, -dx), order=1, mode="nearest")
    for dy, dx in true_shifts
]).astype(np.float32)

rows, crop_rows, crops = estimate_crop_z_registration(
    sim_vol,
    crops_yx=[(20, 105, 25, 110)],
    anchor_z=0,
    max_shift_px=10,
    binning=1,
    highpass_sigma_px=3,
    min_corr=0.05,
    min_overlap_fraction=0.2,
)
recovered = np.array([[r["shift_y_px"], r["shift_x_px"]] for r in rows], dtype=float)
print("true shifts to apply:\n", true_shifts)
print("recovered shifts:\n", np.round(recovered, 2))
print("max abs error:", np.nanmax(np.abs(recovered - true_shifts)))

sim_corr = apply_z_registration(sim_vol, rows)
print("MSE before z2 vs anchor:", np.mean((sim_vol[2] - sim_vol[0])**2))
print("MSE after  z2 vs anchor:", np.mean((sim_corr[2] - sim_corr[0])**2))